In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from datetime import datetime

In [ ]:
from curl_cffi import requests
session = requests.Session(impersonate="chrome")

In [ ]:
# PARAMETERS

TICKER = "AAPL"
START_DATE = "2020-01-01"
END_DATE = "2023-01-01"
LOOKBACK = 20
RSI_PERIOD = 14
MOMENTUM_TYPE = "rsi"
RSI_THRESHOLD = 55
SMA_SLOPE_PERIOD = 10
SMA_SLOPE_MIN = 0.05
STOP_LOSS_PCT = 0.02
TAKE_PROFIT_PCT = 0.04
CAPITAL = 10000
RISK_PCT = 0.02

In [ ]:
def load_data(ticker, start, end):
    df = yf.download(ticker, start=start, end=end, session=session, multi_level_index=False)
    df = df[["Open", "High", "Low", "Close", "Volume"]]
    return df

In [ ]:
def calculate_rsi(series, period = 14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(window=period).mean()
    loss = (-delta.clip(upper=0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi 

In [ ]:
def calculate_sma_slope(series, period=10):
    sma = series.rolling(window=period).mean()
    slope = sma.diff()
    return slope

In [ ]:
def generate_signals(df, lookback, momentum_type, rsi_period, rsi_threshold, sma_slope_period, sma_slope_min):
    close_series = df["Close"].squeeze()
    df["rolling_high"] = df["High"].rolling(window=lookback).max().shift(1)
    df["rolling_low"] = df["Low"].rolling(window=lookback).min().shift(1)
    df["rsi"] = calculate_rsi(close_series, rsi_period)
    df["sma_slope"] = calculate_sma_slope(close_series, sma_slope_period)
    df = df.dropna().copy()

    close = pd.Series(np.ravel(df["Close"].values), index=df.index)
    rolling_high = pd.Series(np.ravel(df['rolling_high'].values), index=df.index)
    rolling_low = pd.Series(np.ravel(df['rolling_low'].values), index=df.index)
    
    close, rolling_high = close.align(rolling_high, join="inner")
    close, rolling_low = close.align(rolling_low, join="inner")

    df["long_signal"] = (close > rolling_high)
    df["short_signal"] = (close < rolling_low)

    if momentum_type == "rsi": 
        df["long_signal"] &= (df["rsi"] > rsi_threshold)
        df["short_signal"] &= (df["rsi"] < (100 - rsi_threshold))
    elif momentum_type == 'sma_slope':
        df['long_signal'] &= (df['sma_slope'] > sma_slope_min)
        df['short_signal'] &= (df['sma_slope'] < -sma_slope_min)

    return df

In [ ]:
def backtest(df, stop_loss_pct, take_profit_pct, capital, risk_pct):
    trades = []
    equity = [capital]
    position = None
    entry_price = entry_time = size = stop_loss = take_profit = None

    for i in range(1, len(df)):
        row = df.iloc[i]
        close = float(row["Close"])
        high = float(row["High"])
        low = float(row["Low"])


        long_sig = row["long_signal"]
        short_sig = row["short_signal"]
        if isinstance(long_sig, (pd.Series, np.ndarray)):
            long_sig = bool(long_sig.item())
        if isinstance(short_sig, (pd.Series, np.ndarray)):
            short_sig = bool(short_sig.item())
        long_sig  = bool(long_sig)
        short_sig = bool(short_sig)

        if position is None:
            if long_sig or short_sig:
                position = "long" if long_sig else "short"
                entry_price = close
                entry_time = row.name
                size = (capital * risk_pct) / (stop_loss_pct * close)

                if position == "long":
                    stop_loss = entry_price * ( 1- stop_loss_pct)
                    take_profit = entry_price * (1 + take_profit_pct)
                else:
                    stop_loss = entry_price * ( 1 + stop_loss_pct)
                    take_profit = entry_price * ( 1 - take_profit_pct)
        
        else:

            exit_price = None
            exit_reason = None

            if position == "long":
                if low <= stop_loss:
                    exit_price, exit_reason = stop_loss, "stop_loss"
                elif high >= take_profit:
                    exit_price, exit_reason = take_profit, "take_profit"
                elif short_sig:
                    exit_price, exit_reason = close, "reverse"

            else: #short
                if high >= stop_loss:
                    exit_price, exit_reason = stop_loss, 'stop_loss'
                elif low <= take_profit:
                    exit_price, exit_reason = take_profit, 'take_profit'
                elif long_sig:
                    exit_price, exit_reason = close, 'reverse'


            if exit_price is not None:
                pnl = (exit_price - entry_price) * size if position == "long" else (entry_price - exit_price) * size
                capital += pnl
                trades.append({
                    'entry_time': entry_time,
                    'exit_time' : row.name,
                    'entry_price': entry_price,
                    'exit_price': exit_price,
                    'position': position,
                    'pnl': pnl,
                    'exit_reason': exit_reason,
                })
                position = entry_price = entry_time = size = None
                stop_loss = take_profit = None

        if position == 'long':
            equity.append(capital + (close - entry_price) * size)
        elif position == 'short':
            equity.append(capital + (entry_price - close) * size)
        else:
            equity.append(capital)
 
    return trades, equity

In [ ]:
def compute_metrics(trades, equity):
    if not trades:
        return {"win_rate" : 0, "profit_factor": 0, "max_drawdown": 0}

    wins = [t["pnl"] for t in trades if t["pnl"] > 0]
    losses = [- t["pnl"] for t in trades if t["pnl"] < 0]

    win_rate = len(wins) / len(trades)
    profit_factor = sum(wins) / sum(losses) if losses else float('inf')

    peak = equity[0]
    max_drawdown = 0
    for val in equity:
        peak = max(peak, val)
        max_drawdown = min(max_drawdown, (val - peak) / peak)
        
    return {'win_rate': win_rate, 'profit_factor': profit_factor, 'max_drawdown': max_drawdown}

In [ ]:
def plot_results(df, trades, equity):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
 
    ax1.plot(df.index, df['Close'],        label='Close Price')
    ax1.plot(df.index, df['rolling_high'], label='Rolling High', linestyle='--', alpha=0.7)
    ax1.plot(df.index, df['rolling_low'],  label='Rolling Low',  linestyle='--', alpha=0.7)
    if trades:
        trades_df = pd.DataFrame(trades)
        ax1.scatter(trades_df['entry_time'], trades_df['entry_price'],
                    color='green', label='Entry', zorder=5)
        ax1.scatter(trades_df['exit_time'], trades_df['exit_price'],
                    color='red', label='Exit', zorder=5)
    ax1.legend()
    ax1.set_title('Price & Signals')
    ax1.set_ylabel('Price')
 
    if MOMENTUM_TYPE == 'rsi':
        ax2.plot(df.index, df['rsi'], label='RSI', color='purple')
        ax2.axhline(RSI_THRESHOLD,       linestyle='--', color='gray', alpha=0.7)
        ax2.axhline(100 - RSI_THRESHOLD, linestyle='--', color='gray', alpha=0.7)
        ax2.set_title('RSI')
        ax2.set_ylabel('RSI')
    else:
        ax2.plot(df.index, df['sma_slope'], label='SMA Slope', color='orange')
        ax2.axhline(0, linestyle='--', color='gray', alpha=0.7)
        ax2.set_title('SMA Slope')
        ax2.set_ylabel('Slope')
    ax2.legend()
 
    plt.tight_layout()
    plt.show()

In [ ]:
def main():
    df = load_data(TICKER, START_DATE, END_DATE)
    print('Columns after load:', df.columns.tolist())
    print('Shape:', df.shape)
    print('Head:\n', df.head(3))
    
    df = generate_signals(df, LOOKBACK, MOMENTUM_TYPE, RSI_PERIOD, RSI_THRESHOLD, SMA_SLOPE_PERIOD, SMA_SLOPE_MIN)
    print('Bars after dropna:', len(df))
    print('Long signals:', df['long_signal'].sum())
    print('Short signals:', df['short_signal'].sum())
    
    trades, equity = backtest(df, STOP_LOSS_PCT, TAKE_PROFIT_PCT, CAPITAL, RISK_PCT)
    print('Number of trades:', len(trades))
    
    metrics = compute_metrics(trades, equity)
    print('Win Rate:     ', round(metrics['win_rate'] * 100, 2), '%')
    print('Profit Factor:', round(metrics['profit_factor'], 2))
    print('Max Drawdown: ', round(metrics['max_drawdown'] * 100, 2), '%')
    plot_results(df, trades, equity)
    print('Backtest complete. Close the chart window to finish.')

In [ ]:
if __name__ == "__main__":
    main()